In [1]:
# mount google drive
from google.colab import drive 
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# import cell for all the imports i am using during the execuiton
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from itertools import combinations
import os
import sys
import os
import torch 
import re
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
import pandas as pd
from collections import defaultdict
from pathlib import Path
import csv

In [3]:
%%bash
# Clonning elasticface and getting the pre-trained model checkpoint
# this one is bash script with strict exit on error

set -e

# clonning elasticface repo if it's not cloned already
if [ ! -d "ElasticFace" ]; then
    git clone "https://github.com/fdbtrs/ElasticFace"
fi

# installing gdown to get stuff from google drive
pip install -q gdown
# downloading the pre-trained model and doing some cleanup cause unfrotunately google drive just works that way
mkdir -p checkpoints

# load both backbone files and just rename them accordingly
# load face arc checkpoint 
gdown 'https://drive.google.com/drive/folders/1q3ws_BQLmgXyiy2msvHummXq4pRqc1rx' --folder -O checkpoints
rm -f checkpoints/training.log
mv checkpoints/*.pth checkpoints/ElasticFaceArc.pth 2>/dev/null || true

# do the same for cos 
gdown 'https://drive.google.com/drive/folders/1ZiLLZXQ1jMzFwMGhYjtMwcdHmuedQb-2' --folder -O checkpoints
rm -f checkpoints/training.log
mv checkpoints/*.pth checkpoints/ElasticFaceCos.pth 2>/dev/null || true


ls "checkpoints/"


echo "Setup completed successfully"

Processing file 1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb 295672backbone.pth
Processing file 1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH training.log
Processing file 1djUe3JWHwJILk1dprwZWer0OdjtJos48 295672backbone.pth
Processing file 1WT1ISIdI8jFsOsxLRF_SrX12o-LDE_Qv config.py
Processing file 1XgfEQgEabinH--VhIusWQ8Js43vz1vK0 training.log
295672backbone.pth
config.py
ElasticFaceArc.pth
Setup completed successfully


Cloning into 'ElasticFace'...
Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb
From (redirected): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb&confirm=t&uuid=337c24d6-e135-4d71-86dd-6bdb11429d7d
To: /content/checkpoints/295672backbone.pth
100%|██████████| 261M/261M [00:07<00:00, 34.1MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH
To: /content/checkpoints/training.log
100%|██████████| 788k/788k [00:00<00:00, 6.44MB/s]
Download completed
Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1djUe3JWHwJILk1dprwZWer0OdjtJos48
From (redirected): https://drive.google.com/uc?id=1djUe3JWHwJILk1dprwZWer0OdjtJos48&co

In [ ]:
# -------------------------------------------------------------
# 1. Corrected Masking Function (Channel-wise DFT as in Section 3.2)
# -------------------------------------------------------------
def create_sharp_mask(image_path: str, masking_range: tuple[int, int], target_size=(112, 112)):
    # Load image and resize to model's expected 112x112 input
    img = cv2.imread(image_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Ensure RGB order
    
    # establish image center
    h, w, _ = img.shape
    crow, ccol = h // 2, w // 2
    
    # Compute L2 radial distance matrix from center
    y, x = np.ogrid[:h, :w]
    radius = np.sqrt((x - ccol)**2 + (y - crow)**2)
    
    # 2D Mask: 0 inside [r_min, r_max], 1 elsewhere
    mask_2d = np.ones((h, w), dtype=np.float32)
    mask_2d[(radius >= masking_range[0]) & (radius <= masking_range[1])] = 0.0
    
    # Channel-wise DFT (OpenCV expects 1-channel for 2D real DFT)
    channels = cv2.split(np.float32(img))
    masked_channels = []
    for ch in channels:
        dft = cv2.dft(ch, flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)
        dft_masked = dft_shift * mask_2d[:, :, np.newaxis] # Zero target band
        dft_ishift = np.fft.ifftshift(dft_masked)
        rec_ch = cv2.idft(dft_ishift, flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
        masked_channels.append(rec_ch)
        
    reconstructed_img = cv2.merge(masked_channels)
    
    # Convert to PyTorch Tensor and normalize to [-1, 1] (ElasticFace standard)
    tensor_img = torch.from_numpy(reconstructed_img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0) # Shape: (1, 3, 112, 112)

def preprocess_unmasked_image(image_path: str, target_size=(112, 112)):
    """Preprocess the unmasked trusted probe image identically."""
    img = cv2.imread(image_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor_img = torch.from_numpy(img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0) # Shape: (1, 3, 112, 112) ALREADY TENSOR

In [5]:
def create_blur_mask(
    image_path: str,
    masking_range: tuple[int, int],
    blur_kernel_size: int,
    target_size=(112, 112),
):
    if len(masking_range) != 2:
        raise ValueError(
            "masking_range must contain exactly two values."
        )

    if masking_range[0] < 0 or masking_range[1] < masking_range[0]:
        raise ValueError(
            "masking_range must be a non-negative "
            "(min_radius, max_radius) tuple."
        )

    if blur_kernel_size <= 0 or blur_kernel_size % 2 == 0:
        raise ValueError(
            "blur_kernel_size must be a positive odd integer."
        )

    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(
            f"Could not read image: {image_path}"
        )

    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape
    crow, ccol = h // 2, w // 2

    y, x = np.ogrid[:h, :w]
    radius = np.sqrt((x - ccol) ** 2 + (y - crow) ** 2)

    mask_2d = np.ones((h, w), dtype=np.float32)
    mask_2d[
        (radius >= masking_range[0])
        & (radius <= masking_range[1])
    ] = 0.0

    # Blur the frequency-domain mask ITSELF BEFORE APPLYING TO THE IMAGE
    mask_2d = cv2.GaussianBlur(
        mask_2d,
        (blur_kernel_size, blur_kernel_size),
        sigmaX=0,
    )

    channels = cv2.split(np.float32(img))
    reconstructed_channels = []

    for channel in channels:
        dft = cv2.dft(channel, flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)

        dft_masked = dft_shift * mask_2d[:, :, np.newaxis] # target band will be faded not completely zeroed out

        dft_ishift = np.fft.ifftshift(dft_masked)
        reconstructed_channel = cv2.idft(
            dft_ishift,
            flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT,
        )

        reconstructed_channels.append(reconstructed_channel)

    reconstructed_img = cv2.merge(reconstructed_channels)
    reconstructed_img = np.clip(
        reconstructed_img,
        0,
        255,
    ).astype(np.uint8)

    tensor_img = torch.from_numpy(reconstructed_img)
    tensor_img = tensor_img.permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5

    return tensor_img.unsqueeze(0)

In [ ]:
# Loading the model 

def load_elasticface_model(checkpoint_path: str):
    sys.path.append("./ElasticFace")
    from backbones.iresnet import iresnet100

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = iresnet100().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint)
    model.eval()

    fake_input = torch.randn(1, 3, 112, 112, device=device)
    with torch.inference_mode():
        fake_output = model(fake_input)
    print(f"Fake output shape: {fake_output.shape}, if that works, the model is loaded and working correctly.")

    return model, device

In [7]:
# Returns a list of dicts, one per morph:
# [{"morph": morph_path, "verification1": image_path, "verification2": image_path}, ...]
# The exact source probe names are extracted from the morph filename and excluded,
# while the remaining same-identity images are used as verification probes.
def get_evaluation_probes(image_files: list[str], morphs_path: str) -> tuple[list[dict], tuple[str, ...]]:
    results = []
    bad_probes = []

    for morph in sorted(os.listdir(morphs_path)):
        if not morph.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        # getting the full path to the morph and it's name without the extension
        morph_path = os.path.join(morphs_path, morph)
        morph_name = os.path.splitext(os.path.basename(morph))[0]

        # using regex, extracting the identities which were used to construct the morph
        identity_1_name = re.search(r"(?<=^morphed_).*(?=_and)", morph_name).group(0)
        identity_2_name = re.search(r"(?<=_and_).*(?=$)", morph_name).group(0)

        # adding those identities to the BAD_PROBES tuple to avoid data leakage
        bad_probes.extend((identity_1_name, identity_2_name))

        # getting identity prefixes to find other probes of the same identity
        identity_1_prefix = identity_1_name.split("_")[0]
        identity_2_prefix = identity_2_name.split("_")[0]

        # getting the paths to the other probes of the identities, excluding the morph identity itself
        identity_1_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_1_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_1_name
        )
        identity_2_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_2_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_2_name
        )

        if len(identity_1_probes) < 2 or len(identity_2_probes) < 2:
            continue

        # getting first probe of each identity to use for verification
        results.append({
            "morph": morph_path,
            "verification1": identity_1_probes[0],
            "verification2": identity_2_probes[0],
            "fnmr_stash": [identity_1_probes[1], identity_2_probes[1]] # since we are having 3 photos per identity, stashing the third already not to find them again after
        })

    return results, tuple(bad_probes)


In [8]:
# Functions for batch processing images for FMR calibration and FNMR calculation later.
def build_tensor(image_path: str, masked: bool):
    if not masked:
        return preprocess_unmasked_image(image_path)

    match MASKING_METHOD:
        case "none":
            # Baseline: unmasked reference photo.
            return preprocess_unmasked_image(image_path)

        case "sharp":
            # MR-FR: mask the reference photo.
            return create_sharp_mask(image_path, masking_range)

        case "blur":
            # MR-FR: mask and blur the reference photo.
            return create_blur_mask(
                image_path, masking_range, blur_kernel_size
            )

        case _:
            raise ValueError(f"Unknown masking method: {MASKING_METHOD}")


def compute_embeddings(model, device, paths, masked, batch_size=32):
    """Compute normalized embeddings while processing at most 32 images per batch."""
    embeddings = []

    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        batch = torch.cat([
            build_tensor(path, masked)
            for path in batch_paths
        ]).to(device)

        with torch.inference_mode():
            output = model(batch)
            output = F.normalize(output, p=2, dim=1)

        embeddings.append(output)

    if not embeddings:
        return torch.empty((0, 0), device=device)

    return torch.cat(embeddings)

In [ ]:
# Perform calibration for the requested FMR value and save the threshold.
# The threshold is reused for the later FNMR match/not-match decisions.

# Build a dictionary containing one calibration image per identity.
def get_calibration_dict(image_files, bad_probes):
    calibration_dict = {}
    for image_path in image_files:
        identity_name = image_path.split("/")[-2]
        if identity_name in calibration_dict:
            continue

        # Exclude identities used to create morphs to avoid data leakage.
        if os.path.splitext(os.path.basename(image_path))[0] in bad_probes:
            continue

        calibration_dict[identity_name] = image_path
    return calibration_dict




def get_calibration_threshold(model, device, calibration_dict) -> float:
    model.eval()
    calibration_threshold = None

    # compute_embeddings expects image paths, while calibration_dict keys are identities.
    calibration_paths = list(calibration_dict.values())
    print(f"Calibrating on {len(calibration_paths)} identities...")

    # The first image is processed according to MASKING_METHOD.
    masked_embeddings = compute_embeddings(model, device, calibration_paths, masked=True)

    # The second image is always an unmasked live probe.
    unmasked_embeddings = compute_embeddings(model, device, calibration_paths, masked=False)

    # Every row/column pair gives one impostor score.
    score_matrix = masked_embeddings @ unmasked_embeddings.T

    # Keep only different-identity comparisons.
    upper_triangle = torch.triu(
        torch.ones_like(score_matrix, dtype=torch.bool),
        diagonal=1,
    )

    impostor_scores = score_matrix[upper_triangle].detach().cpu().numpy()
    impostor_scores = np.asarray(impostor_scores)

    print(f"Total impostor comparisons computed: {len(impostor_scores)}")

    percentile = (1.0 - TARGET_FMR) * 100.0
    calibration_threshold = float(np.percentile(impostor_scores, percentile))
    print(
        f"Calibration completed. Threshold for FMR={TARGET_FMR}: "
        f"{calibration_threshold:.4f}"
    )
    
    return calibration_threshold

In [ ]:
def get_identity(image_path: str) -> str:
    return Path(image_path).parent.name



def get_live_legit_probes(image_files: list[str], evaluation_probes: list[dict]):   
    # Group every dataset image by identity.
    images_by_identity = defaultdict(list)

    # get dictionary of all images per identity where key is identity name 
    for image_path in image_files:
        identity = get_identity(image_path)
        images_by_identity[identity].append(image_path)


    # Preserve the probes selected for morph evaluation.
    live_by_identity = defaultdict(set)
    legit_by_identity = defaultdict(set)

    for morphtriplet in evaluation_probes:
        # getting verification paths for the morph
        verification_paths = [
            morphtriplet[f"verification{i}"]
            for i in range(1, 3)
        ]

        # getting "key for the image"
        for image_path in verification_paths:
            identity = get_identity(image_path)
            live_by_identity[identity].add(image_path) # add them to respective live identities

        # do the same for the stashed things we haven't used anywhere
        for image_path in morphtriplet["fnmr_stash"]:
            identity = get_identity(image_path)
            legit_by_identity[identity].add(image_path)


    # Add identities that were never used in a morph.
    # For these identities, use one image as a live probe and one as a legit probe.
    # morph identities is a set union -- so all identity names without duplicates
    morph_identities = set(live_by_identity) | set(legit_by_identity)

    for identity, paths in images_by_identity.items():
        if identity in morph_identities:
            continue # if identity is already one of the morphed, it's probes are already added and we don't need it second time

        # if that's not the case, just add first 2 probes for this identity
        paths = sorted(paths)

        if len(paths) >= 2:
            live_by_identity[identity].add(paths[0])
            legit_by_identity[identity].add(paths[1])

    return live_by_identity, legit_by_identity

In [ ]:
# Calculate FNMR using identity-preserving legit/live probe pairs.
# Each legit image is compared only with live images from the same identity.
def calculate_fnmr(model, device, live_by_identity, legit_by_identity, calibration_threshold) -> float:

    if calibration_threshold is None:
        raise RuntimeError("calibration_threshold must be calculated before FNMR evaluation.")

    # Keep only identities that have both kinds of probe available.
    matching_identities = sorted(
        set(live_by_identity) & set(legit_by_identity)
    )

    # Build every genuine comparison for each identity. Sets remove duplicate paths;
    # sorting makes the evaluation order deterministic.
    comparison_pairs = [
        (identity, legit_path, live_path)
        for identity in matching_identities
        for legit_path in sorted(legit_by_identity[identity])
        for live_path in sorted(live_by_identity[identity])
    ]

    if not comparison_pairs:
        raise RuntimeError("No same-identity legit/live pairs are available for FNMR.")

    # Compute each unique image embedding once, then reuse it for all pairs.
    legit_paths = sorted({legit_path for _, legit_path, _ in comparison_pairs})
    live_paths = sorted({live_path for _, _, live_path in comparison_pairs})

    # Legit passport probes are processed with the selected masking method.
    legit_embeddings = compute_embeddings(
        model,
        device,
        legit_paths,
        masked=True,
        batch_size=32,
    )

    # Live capture probes are always processed without masking.
    live_embeddings = compute_embeddings(
        model,
        device,
        live_paths,
        masked=False,
        batch_size=32,
    )

    legit_embedding_by_path = {
        path: embedding
        for path, embedding in zip(legit_paths, legit_embeddings)
    }
    live_embedding_by_path = {
        path: embedding
        for path, embedding in zip(live_paths, live_embeddings)
    }

    # Normalized embeddings make the dot product equal to cosine similarity.
    fnmr_rows = []
    for identity, legit_path, live_path in comparison_pairs:
        similarity = float(
            torch.dot(
                legit_embedding_by_path[legit_path],
                live_embedding_by_path[live_path],
            )
        )
        is_match = similarity >= calibration_threshold

        fnmr_rows.append({
            "identity": identity,
            "legit_path": legit_path,
            "live_path": live_path,
            "similarity": similarity,
            "match": is_match,
            "decision": "match" if is_match else "not_match",
        })

    fnmr_results_df = pd.DataFrame(fnmr_rows)
    false_non_matches = (~fnmr_results_df["match"]).sum()
    total_genuine_comparisons = len(fnmr_results_df)
    fnmr = false_non_matches / total_genuine_comparisons

    print(f"Identities evaluated: {len(matching_identities)}")
    print(f"Genuine comparisons: {total_genuine_comparisons}")
    print(f"False non-matches: {false_non_matches}")
    print(f"FNMR: {fnmr:.6f} ({fnmr:.2%})")
    return fnmr

In [ ]:
# 2 funcitons which are used to determine how similar are the morphs to original people 
# first one gets the embedding of the image 
# second calculates the cosine similarity between two embeddings
def get_embedding(model, image_path: str, device) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform_pipeline(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model(image_tensor)

    if isinstance(embedding, (tuple, list)):
        embedding = embedding[0]

    # keep it on gpu
    embedding = embedding.detach().float()

    if embedding.ndim > 1:
        embedding = embedding.reshape(embedding.shape[0], -1)
        if embedding.shape[0] == 1:
            embedding = embedding[0]

    return embedding


def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    a = a.view(-1)
    b = b.view(-1)
    denom = (torch.norm(a) * torch.norm(b)).clamp_min(1e-12)
    return float(torch.dot(a, b) / denom)

In [13]:
def mmpmr_experiment(model, device, evaluation_probes, dataset_name, calibration_threshold):
    model.eval() # make sure the model is in eval mode

    match MASKING_METHOD: 
        case "none":
            CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{dataset_name}_cosine_similarity_{MASKING_METHOD}_{os.getenv('FR_MODEL_NAME')}.csv"
        case "sharp": 
            CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{dataset_name}_cosine_similarity_{MASKING_METHOD}_{masking_range[0]}_{masking_range[1]}_{os.getenv('FR_MODEL_NAME')}.csv"
        case "blur": 
            CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{dataset_name}_cosine_similarity_{MASKING_METHOD}_{masking_range[0]}_{masking_range[1]}_{blur_kernel_size}_{os.getenv('FR_MODEL_NAME')}.csv"


    os.makedirs(os.path.dirname(CSV_OUTPUT_PATH), exist_ok=True)


    rows = []
    fooled_counter = 0
    for item in evaluation_probes:
        morph_path = item["morph"]
        verification1_path = item["verification1"]
        verification2_path = item["verification2"]

        if morph_path is None or verification1_path is None or verification2_path is None:
            continue

        # siwtch on how are we going to process the morph based on the selected masking method 
        match MASKING_METHOD:
            case "none":
                morph_embedding = get_embedding(model, morph_path, device)
            case "sharp":
                morph_masked_tensor = create_sharp_mask(morph_path, masking_range).to(device)
                with torch.no_grad():
                    morph_embedding = model(morph_masked_tensor)
            case "blur":
                    morph_masked_tensor = create_blur_mask(
                        morph_path, masking_range, blur_kernel_size
                    ).to(device)
                    with torch.no_grad():
                        morph_embedding = model(morph_masked_tensor)
            case _:
                raise ValueError(f"Unknown masking method: {MASKING_METHOD}")
            
        verification1_embedding = get_embedding(model, verification1_path, device)
        verification2_embedding = get_embedding(model, verification2_path, device)

        sim1 = cosine_similarity(morph_embedding, verification1_embedding)
        sim2 = cosine_similarity(morph_embedding, verification2_embedding)

        fooled = False 

        if sim1 >= calibration_threshold and sim2 >= calibration_threshold:
            fooled = True
            fooled_counter += 1 

        rows.append({
            "morph": morph_path,
            "verification1": verification1_path,
            "verification2": verification2_path,
            "1_similarity": sim1,
            "2_similarity": sim2,
            "fooled": fooled
        })

    results_df = pd.DataFrame(rows, columns=["morph", "verification1", "verification2", "1_similarity", "2_similarity", "fooled"])
    results_df.to_csv(CSV_OUTPUT_PATH, index=False)

    print(f"Saved {len(results_df)} rows to: {CSV_OUTPUT_PATH}")
    print(results_df.head())


    # calculating MMPMR (Morph Match Presentation Match Rate) for the dataset
    MMPMR = round(results_df["fooled"].value_counts().get(True)/len(results_df),2)
    print(f"MMPMR for this dataset is: {MMPMR}")
    return MMPMR

In [ ]:
# create a csv file if not exists and put the header in it
# then just add results to it.

def save_experiment_result(
    dataset_name: str,
    morphing_model: str,
    fr_model: str,
    masking_method: str,
    masking_range: tuple[int, int] | None,
    blur_kernel_size: int | None,
    target_fmr: float,
    fnmr: float,
    mmpmr: float,
    results_csv_path: str
) -> None: # writes the experiemnt result to the csv and creates it if it doesn't exist 

    row = [
        dataset_name,
        morphing_model,
        fr_model,
        masking_method,
        masking_range[0] if masking_range else "",
        masking_range[1] if masking_range else "",
        blur_kernel_size if MASKING_METHOD == "blur" else "",
        target_fmr,
        fnmr,
        mmpmr,
    ]

    results_path = Path(results_csv_path)
    file_exists = results_path.exists()

    with results_path.open("a+", newline="") as f:
        f.seek(0) # Move to the beginning of the file to read existing rows
        existing_rows = list(csv.reader(f))

        # Compare only the experiment configuration, excluding result metrics.
        experiment_key = tuple(row[:8])
        already_exists = any(tuple(existing[:6]) == experiment_key for existing in existing_rows)

        if not already_exists:
            f.seek(0, 2)  # Move to the end before writing
            writer = csv.writer(f)

            if not file_exists or not existing_rows:
                writer.writerow([
                    "dataset",
                    "morphing_model",
                    "fr_model",
                    "masking_method",
                    "masking_range_start",
                    "masking_range_end",
                    "blur_kernel_size",
                    "target_fmr",
                    "fnmr",
                    "mmpmr",
                ])

            writer.writerow(row)
            print("New experiment written.")
        else:
            print("Experiment already exists; skipping.")

In [15]:
# Constants block

# paths and names 
# DATASET_NAME = "HR_128_normalized" # VGGFace2_filtered_10_first300 filtered_dataset_ferret_3 HR_128_normalized
# MORPHING_MODEL = "face_morpher" # either "face_morpher" or "mor_diff"
# ALIGNED_DATASET_PATH = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_NAME}"
# MORPHS_PATH = f"/content/drive/MyDrive/Morphs/{MORPHING_MODEL}/{DATASET_NAME}"
results_csv_path = '/content/drive/MyDrive/Morph_Evaluation/results_summary.csv'

# FR model to use (setting as environmental variable cause it's used by python block later)
os.environ["FR_MODEL_NAME"] = "ElasticFaceArc" # or ElasticFaceCos

# masking parameters
MASKING_METHOD = "blur" # either "sharp" or "blur" or "none" for the control group
BAD_PROBES = () # here probes used for morphing would be added, as they shal not be used anywhere else to avoid data leakage
TARGET_FMR = 0.01 # target false match rate for evaluation. As per EU standards it can be either a 0.1% or 1% FMR, so either 0.001 or 0.01
# blur_kernel_size = 5 # provide an odd integer for the kernel size

In [ ]:
# Load the model once, then reuse it for every dataset.
model, device = load_elasticface_model(f"./checkpoints/{os.getenv('FR_MODEL_NAME')}.pth")

transform_pipeline = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Add or remove datasets here. Morphs are expected under the selected MORPHING_MODEL.
DATASET_NAMES = [
    "HR_128_normalized",
    "VGGFace2_filtered_10_first300",
    "filtered_dataset_ferret_3"
]

MORPHING_MODELS = ["face_morpher", "mor_diff"]

masking_step = 3
# create a list of masking ranges if masking method is sharp or blur, else just none
if MASKING_METHOD in {"sharp", "blur"}:
    masking_ranges = [
        (masking_start, masking_start + masking_step)
        for masking_start in range(0, 12, masking_step)
    ]

    if MASKING_METHOD == "blur":
        blur_kernel_sizes = [3,5,7,9,11]
else:
    # One experiment with no masking range
    masking_ranges = [None]
    blur_kernel_sizes = [None]

# if none it will just do one iteration allowing to run control group
# without repeating itself
for masking_range in masking_ranges:
    
    # if method is "sharp" then the following masking parameters should be provided 
    if MASKING_METHOD == "sharp" and not masking_range:
        raise ValueError("Something went wrong with the masking range.")

    # same for blur kernel sizes
    for blur_kernel_size in blur_kernel_sizes:

        # if method is blur then the following masking parameters should be provided
        if MASKING_METHOD == "blur" and (not masking_range or blur_kernel_size is None):
            raise ValueError("Something went wrong with blur kernel size assignmnent or masking range.")

        for morphing_model in MORPHING_MODELS:
            MORPHING_MODEL = morphing_model
            
            for dataset_name in DATASET_NAMES:
                DATASET_NAME = dataset_name
                ALIGNED_DATASET_PATH = f"/content/drive/MyDrive/Datasets_aligned/{dataset_name}"
                MORPHS_PATH = f"/content/drive/MyDrive/Morphs/{MORPHING_MODEL}/{dataset_name}"

                print(f"""Running experiment for dataset: {dataset_name},
                            morphing model: {MORPHING_MODEL},
                            masking method: {MASKING_METHOD},
                            masking range: {masking_range},
                            blur kernel size: {blur_kernel_size}""")

                experiment_config = [dataset_name, MORPHING_MODEL, os.getenv("FR_MODEL_NAME"), MASKING_METHOD, masking_range[0], masking_range[1], blur_kernel_size, TARGET_FMR]

                with open(results_csv_path, "a+", newline="") as f:
                    f.seek(0)
                    existing_rows = list(csv.reader(f))
                    experiment_key = tuple("" if value is None else str(value)
                           for value in experiment_config)
                    already_exists = any(tuple(existing[:8]) == experiment_key for existing in existing_rows)

                    if already_exists:
                        print("Experiment already exists; skipping.")
                        continue

                # Reset tuple for identities used during morphing 
                BAD_PROBES = ()

                image_files = sorted([
                    os.path.join(root, filename)
                    for root, _, files in os.walk(ALIGNED_DATASET_PATH)
                    for filename in files
                    if filename.lower().endswith((".png", ".jpg", ".jpeg"))
                ])
                if not image_files:
                    raise FileNotFoundError(f"No images found in {ALIGNED_DATASET_PATH}")

                evaluation_probes, BAD_PROBES = get_evaluation_probes(
                    image_files=image_files,
                    morphs_path=MORPHS_PATH,
                )
                if not evaluation_probes:
                    raise RuntimeError(f"No evaluation probes found in {MORPHS_PATH}")

                # calibration dict is basically dict of all probes per identity minus those which were used to create morphs, to avoid data leakage
                calibration_dict = get_calibration_dict(image_files, BAD_PROBES)
                calibration_threshold = get_calibration_threshold(
                    model, device, calibration_dict
                )

                # live by identity are the same pictures which are got from original morph triplets
                # legit are the once used as "real" passport images, which are not used in morphs and live capture
                live_by_identity, legit_by_identity = get_live_legit_probes(
                    image_files, evaluation_probes
                )

                MMPMR = mmpmr_experiment(
                    model=model,
                    device=device,
                    evaluation_probes=evaluation_probes,
                    dataset_name=dataset_name,
                    calibration_threshold=calibration_threshold,
                )
                FNMR = calculate_fnmr(
                    model,
                    device,
                    live_by_identity,
                    legit_by_identity,
                    calibration_threshold,
                )

                save_experiment_result(
                    dataset_name=dataset_name,
                    morphing_model=MORPHING_MODEL,
                    fr_model=os.getenv("FR_MODEL_NAME"),
                    masking_method=MASKING_METHOD,
                    masking_range=masking_range,
                    blur_kernel_size=blur_kernel_size if MASKING_METHOD == "blur" else None,
                    target_fmr=TARGET_FMR,
                    fnmr=FNMR,
                    mmpmr=MMPMR,
                    results_csv_path=results_csv_path,
                )

Using device: cuda
Fake output shape: torch.Size([1, 512]), if that works, the model is loaded and working correctly.
Running experiment for dataset: HR_128_normalized,
                            morphing model: face_morpher,
                            masking method: blur,
                            masking range: (0, 2),
                            blur kernel size: 3
Calibrating on 249 identities...
Total impostor comparisons computed: 30876
Calibration completed. Threshold for FMR=0.01: 0.2163
Saved 118 rows to: /content/drive/MyDrive/Morph_Evaluation/HR_128_normalized_cosine_similarity_blur_0_2_3_ElasticFaceArc.csv
                                               morph  \
0  /content/drive/MyDrive/Morphs/face_morpher/HR_...   
1  /content/drive/MyDrive/Morphs/face_morpher/HR_...   
2  /content/drive/MyDrive/Morphs/face_morpher/HR_...   
3  /content/drive/MyDrive/Morphs/face_morpher/HR_...   
4  /content/drive/MyDrive/Morphs/face_morpher/HR_...   

                                